In [2]:
import json, glob

# Find the notebook wherever it lives
matches = glob.glob("**/pmm_dynamic_multi_exchange_sweep_mexc_nonkyc.ipynb", recursive=True)
print("Found at:", matches)

if matches:
    nb = json.load(open(matches[0]))
    config_src = "".join(nb["cells"][3]["source"])
    print("VALIDATION_CONTROLLER_COMPAT in config cell:", "VALIDATION_CONTROLLER_COMPAT" in config_src)

Found at: ['pmm_dynamic_multi_exchange_sweep_mexc_nonkyc.ipynb']
VALIDATION_CONTROLLER_COMPAT in config cell: False


In [3]:
import json

nb = json.load(open("pmm_dynamic_multi_exchange_sweep_mexc_nonkyc.ipynb"))
config_src = "".join(nb["cells"][3]["source"])

# Insert VALIDATION_CONTROLLER_COMPAT after SEARCH_CONTROLLER_COMPAT
old = "SEARCH_CONTROLLER_COMPAT = False\n"
new = (
    "SEARCH_CONTROLLER_COMPAT = False\n"
    "\n"
    "# Validation controller mode — True = controller-equivalent sliding window for finalist\n"
    "# evaluation (holdout, recent-window, sensitivity). This is intentional: search is fast,\n"
    "# validation is realistic.\n"
    "VALIDATION_CONTROLLER_COMPAT = True\n"
)

if "VALIDATION_CONTROLLER_COMPAT" not in config_src:
    config_src = config_src.replace(old, new, 1)
    nb["cells"][3]["source"] = config_src.splitlines(keepends=True)
    with open("pmm_dynamic_multi_exchange_sweep_mexc_nonkyc.ipynb", "w") as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)
    print("RESTORED VALIDATION_CONTROLLER_COMPAT in config cell")
else:
    print("Already present — no change needed")

# Verify
nb2 = json.load(open("pmm_dynamic_multi_exchange_sweep_mexc_nonkyc.ipynb"))
print("Verified:", "VALIDATION_CONTROLLER_COMPAT" in "".join(nb2["cells"][3]["source"]))

RESTORED VALIDATION_CONTROLLER_COMPAT in config cell
Verified: True


In [4]:
import json, glob

for pattern in [
    "pmm_dynamic_multi_pair_sweep.ipynb",
    "pmm_dynamic_single_pair_sweep_mexc_xmr_usdt.ipynb",
]:
    matches = glob.glob(f"**/{pattern}", recursive=True)
    if matches:
        nb = json.load(open(matches[0]))
        src = "".join(nb["cells"][3]["source"])
        has_it = "VALIDATION_CONTROLLER_COMPAT" in src
        print(f"{pattern}: VALIDATION_CONTROLLER_COMPAT present = {has_it}")
        if not has_it:
            print(f"  ⚠️  ALSO DAMAGED — needs same fix")

pmm_dynamic_multi_pair_sweep.ipynb: VALIDATION_CONTROLLER_COMPAT present = False
  ⚠️  ALSO DAMAGED — needs same fix
pmm_dynamic_single_pair_sweep_mexc_xmr_usdt.ipynb: VALIDATION_CONTROLLER_COMPAT present = False
  ⚠️  ALSO DAMAGED — needs same fix


In [5]:
import json, glob

FIX_BLOCK = (
    "SEARCH_CONTROLLER_COMPAT = False\n"
    "\n"
    "# Validation controller mode — True = controller-equivalent sliding window for finalist\n"
    "# evaluation (holdout, recent-window, sensitivity). This is intentional: search is fast,\n"
    "# validation is realistic.\n"
    "VALIDATION_CONTROLLER_COMPAT = True\n"
)

notebooks = [
    "pmm_dynamic_multi_exchange_sweep_mexc_nonkyc.ipynb",
    "pmm_dynamic_multi_pair_sweep.ipynb",
    "pmm_dynamic_single_pair_sweep_mexc_xmr_usdt.ipynb",
]

for nb_name in notebooks:
    matches = glob.glob(f"**/{nb_name}", recursive=True)
    if not matches:
        print(f"NOT FOUND: {nb_name}")
        continue
    path = matches[0]
    nb = json.load(open(path))
    config_src = "".join(nb["cells"][3]["source"])
    
    if "VALIDATION_CONTROLLER_COMPAT" in config_src:
        print(f"OK (already present): {nb_name}")
        continue
    
    old = "SEARCH_CONTROLLER_COMPAT = False\n"
    if old not in config_src:
        print(f"ERROR: can't find anchor in {nb_name}")
        continue
    
    config_src = config_src.replace(old, FIX_BLOCK, 1)
    nb["cells"][3]["source"] = config_src.splitlines(keepends=True)
    
    with open(path, "w") as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)
    
    # Verify
    nb2 = json.load(open(path))
    verified = "VALIDATION_CONTROLLER_COMPAT" in "".join(nb2["cells"][3]["source"])
    print(f"RESTORED: {nb_name} — verified={verified}")

OK (already present): pmm_dynamic_multi_exchange_sweep_mexc_nonkyc.ipynb
RESTORED: pmm_dynamic_multi_pair_sweep.ipynb — verified=True
RESTORED: pmm_dynamic_single_pair_sweep_mexc_xmr_usdt.ipynb — verified=True
